# B2-022-probabilistic-latent-models — Practice p14 — Solution

**Type:** proof · **Difficulty:** core · **Concepts:** kl-divergence

*45 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

Lemma (Session 2): $\log x\le x-1$ for all $x>0$, with equality only at $x=1$.

### (a) Nonnegativity

*Case 1: some $p_i>0$ has $q_i=0$.* By convention $\mathrm{KL}(P\|Q)=+\infty\ge0$.

*Case 2: $q_i>0$ for every $i\in S=\{i:p_i>0\}$.* Terms with $p_i=0$ are $0$, so $\mathrm{KL}(P\|Q)=\sum_{i\in S}p_i\log(p_i/q_i)$, and every ratio $q_i/p_i$ with $i\in S$ is a positive real. Applying the lemma with $x=q_i/p_i$ and multiplying by $p_i>0$:
$$-\mathrm{KL}(P\|Q)=\sum_{i\in S}p_i\log\frac{q_i}{p_i}\;\overset{(1)}{\le}\;\sum_{i\in S}p_i\Big(\frac{q_i}{p_i}-1\Big)=\sum_{i\in S}q_i-\sum_{i\in S}p_i=\sum_{i\in S}q_i-1\;\overset{(2)}{\le}\;0.$$
Here $\sum_{i\in S}p_i=1$ because $P$ has all its mass on $S$. For (2) we do not need $\sum_{i\in S}q_i=1$: since every $q_i\ge0$, $\sum_{i\in S}q_i\le\sum_{i=1}^mq_i=1$, and that inequality is all the argument requires. Hence $\mathrm{KL}(P\|Q)\ge0$.

### (b) Equality case

($\Leftarrow$) If $P=Q$, every term with $p_i>0$ is $p_i\log1=0$ and the others are $0$ by convention, so $\mathrm{KL}=0$.

($\Rightarrow$) Suppose $\mathrm{KL}(P\|Q)=0$. It is finite, so we are in Case 2, and both inequalities above must be tight:

- **(1) tight.** (1) is a sum of the termwise inequalities $p_i\log(q_i/p_i)\le p_i(q_i/p_i-1)$, each with $p_i>0$; a sum of such inequalities is an equality only if each one is, so the lemma is tight at every $x=q_i/p_i$, which forces $q_i/p_i=1$, i.e. $q_i=p_i$ for every $i\in S$.
- **(2) tight.** $\sum_{i\in S}q_i=1$, so $\sum_{i\notin S}q_i=0$; since $q_i\ge0$, $q_i=0=p_i$ for every $i\notin S$.

Together $q_i=p_i$ for all $i$, i.e. $P=Q$.

### (c) Role of the support

$P=(\tfrac12,\tfrac12,0)$, $Q=(\tfrac13,\tfrac13,\tfrac13)$, $S=\{1,2\}$.
$\mathrm{KL}(P\|Q)=2\cdot\tfrac12\log\frac{1/2}{1/3}+0=\log\tfrac32\approx0.405465$.
The slack in (2) is $1-\sum_{i\in S}q_i=1-\tfrac23=\tfrac13$: $Q$ puts mass $\tfrac13$ outside the support of $P$, which is why the bound only needs $\sum_{i\in S}q_i\le1$, not equality. Inequality (1) has slack too: $\sum_{i\in S}q_i-1-(-\mathrm{KL})=-\tfrac13+\log\tfrac32\approx0.0721$.

## Numeric sanity check

In [ ]:
import math
import numpy as np

def kl(p, q):
    total = 0.0
    for pi, qi in zip(p, q):
        if pi == 0:
            continue
        if qi == 0:
            return math.inf
        total += pi * math.log(pi / qi)
    return total

P = (0.5, 0.5, 0.0)
Q = (1 / 3, 1 / 3, 1 / 3)
kl_c = kl(P, Q)
slack = 1 - sum(q for p, q in zip(P, Q) if p > 0)

rng = np.random.default_rng(20260927)
sweep = []
for _ in range(500):
    p = rng.random(4) * (rng.random(4) > 0.3)
    if p.sum() == 0:
        continue
    p = p / p.sum()
    q = rng.random(4)
    q = q / q.sum()
    support = p > 0
    bound = q[support].sum() - 1           # right-hand side of (1)
    sweep.append((kl(p, q), -bound))
print(kl_c, slack)

### Answer check

In [ ]:
assert abs(kl_c - math.log(1.5)) < 1e-12
assert abs(slack - 1 / 3) < 1e-12
assert all(k >= 0 and k >= b - 1e-12 for k, b in sweep)   # KL >= 1 - sum_S q >= 0
assert kl((0.2, 0.3, 0.5), (0.2, 0.3, 0.5)) == 0.0